<a href="https://colab.research.google.com/github/Nubiga-lima/kyc-risk-model/blob/main/notebooks/01_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KYC Risk Model


In [3]:
import numpy as np
import pandas as pd
import os

def generate_synthetic_kyc_data(n_samples=5000, random_state=42):
    np.random.seed(random_state)
    customer_id = np.arange(1, n_samples + 1)
    age = np.random.randint(18, 85, n_samples)
    country_risk = np.random.choice(
        ["Low", "Medium", "High"],
        size=n_samples,
        p=[0.75, 0.20, 0.05]
    )
    monthly_txn_count = np.random.poisson(lam=20, size=n_samples)
    monthly_txn_volume = np.round(np.random.gamma(shape=2, scale=500, size=n_samples), 2)
    unusual_txn_flag = (monthly_txn_count > 60).astype(int)
    product_risk_score = np.random.choice([1, 2, 3], size=n_samples, p=[0.70, 0.20, 0.10])
    pep_flag = np.random.choice([0, 1], size=n_samples, p=[0.98, 0.02])
    adverse_media_flag = np.random.choice([0, 1], size=n_samples, p=[0.95, 0.05])
    high_risk_jurisdiction = (country_risk == "High").astype(int)

    # -----------------------------
    # Synthetic Risk Label (Target)
    # -----------------------------
    # Weighted risk scoring logic (transparent & explainable)
    risk_score = (
        0.4 * high_risk_jurisdiction +
        0.3 * pep_flag +
        0.2 * adverse_media_flag +
        0.1 * (monthly_txn_count > 50).astype(int)
    )

    # Add random noise to avoid the label being a perfect deterministic
    # function of the input features. This mimics real-world label
    # imperfection and avoids data leakage in downstream modelling,
    # where a model could otherwise just reverse-engineer this formula
    # instead of learning genuine predictive patterns.
    noise = np.random.normal(loc=0, scale=0.15, size=n_samples)
    risk_score_noisy = risk_score + noise
    risk_score_noisy = np.clip(risk_score_noisy, 0, 1.1)

    risk_class = pd.cut(
        risk_score_noisy,
        bins=[-1, 0.2, 0.6, 1.1],
        labels=["Low", "Medium", "High"]
    )

    df = pd.DataFrame({
        "customer_id": customer_id,
        "age": age,
        "country_risk": country_risk,
        "monthly_txn_count": monthly_txn_count,
        "monthly_txn_volume": monthly_txn_volume,
        "unusual_txn_flag": unusual_txn_flag,
        "product_risk_score": product_risk_score,
        "pep_flag": pep_flag,
        "adverse_media_flag": adverse_media_flag,
        "high_risk_jurisdiction": high_risk_jurisdiction,
        "risk_score": risk_score_noisy,
        "risk_class": risk_class
    })
    return df

def save_synthetic_data(df, path="data/processed/synthetic_kyc_data.csv"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved synthetic dataset to: {path}")

In [4]:
df = generate_synthetic_kyc_data()
df.head()

,customer_id,age,country_risk,monthly_txn_count,monthly_txn_volume,unusual_txn_flag,product_risk_score,pep_flag,adverse_media_flag,high_risk_jurisdiction,risk_score,risk_class
0,1,69,Low,23,2273.60,0,1,0,0,0,0.027894,Low
1,2,32,Low,25,1570.80,0,1,0,0,0,0.043825,Low
2,3,78,Low,15,539.06,0,1,0,0,0,0.052781,Low
3,4,38,Low,14,1977.78,0,1,0,0,0,0.224441,Medium
4,5,41,Medium,20,1026.26,0,2,0,0,0,0.000000,Low


In [5]:
save_synthetic_data(df)

Saved synthetic dataset to: data/processed/synthetic_kyc_data.csv


In [6]:
df.shape

(5000, 12)

In [7]:
df['risk_class'].value_counts()

,count
risk_class,
Low,4178
Medium,792
High,30


In [8]:
df.isnull().sum()

,0
customer_id,0
age,0
country_risk,0
monthly_txn_count,0
monthly_txn_volume,0
unusual_txn_flag,0
product_risk_score,0
pep_flag,0
adverse_media_flag,0
high_risk_jurisdiction,0


In [9]:
save_synthetic_data(df)

Saved synthetic dataset to: data/processed/synthetic_kyc_data.csv
